# Model Comparison: FashionNet vs YOLOv8n
Visual analysis of the two models for the project report.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load comparison results (run compare_models.py first)
with open('../docs/comparison.json') as f:
    data = json.load(f)

custom = data['fashionnet']
yolo   = data['yolov8n']

CATEGORIES = [
    'short_sleeve_top', 'long_sleeve_top', 'short_sleeve_outwear',
    'long_sleeve_outwear', 'vest', 'sling', 'shorts', 'trousers',
    'skirt', 'short_sleeve_dress', 'long_sleeve_dress', 'vest_dress', 'sling_dress'
]
print('Results loaded.')
print(f'FashionNet  mAP@50: {custom["map50"]:.4f}')
print(f'YOLOv8n     mAP@50: {yolo["map50"]:.4f}')

## 1. Overall metrics summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Model Comparison Summary', fontsize=15, fontweight='bold')

metrics = [
    ('mAP@50', 'map50',    True),
    ('FPS',    'fps',      True),
    ('Params (M)', None,   False),
]

colors = ['#4C72B0', '#DD8452']

for ax, (title, key, _) in zip(axes, metrics):
    if key:
        vals = [custom[key], yolo[key]]
    else:
        vals = [custom['params']/1e6, yolo['params']/1e6]
    bars = ax.bar(['FashionNet', 'YOLOv8n'], vals, color=colors, width=0.5, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                f'{val:.3f}' if val < 10 else f'{val:.1f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/comparison_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Per-class mAP@50 comparison

In [ ]:
x      = np.arange(len(CATEGORIES))
width  = 0.38

c_aps = [custom['ap_per_class'].get(str(i), float('nan')) for i in range(13)]
y_aps = [yolo['ap_per_class'].get(str(i), float('nan'))   for i in range(13)]

fig, ax = plt.subplots(figsize=(16, 6))
b1 = ax.bar(x - width/2, c_aps, width, label='FashionNet (custom)', color='#4C72B0', alpha=0.85)
b2 = ax.bar(x + width/2, y_aps, width, label='YOLOv8n (fine-tuned)', color='#DD8452', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in CATEGORIES], fontsize=8)
ax.set_ylabel('AP@50')
ax.set_title('Per-Class AP@50: FashionNet vs YOLOv8n', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(y=custom['map50'], color='#4C72B0', linestyle='--', alpha=0.5, label=f'FashionNet mAP')
ax.axhline(y=yolo['map50'],   color='#DD8452', linestyle='--', alpha=0.5, label=f'YOLOv8n mAP')
ax.spines[['top','right']].set_visible(False)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('../docs/per_class_ap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Speed vs Accuracy trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

models = [
    ('FashionNet', custom['fps'], custom['map50'], '#4C72B0', 120),
    ('YOLOv8n',   yolo['fps'],   yolo['map50'],   '#DD8452', 120),
]

for name, fps, mAP, color, size in models:
    ax.scatter(fps, mAP, s=size*3, color=color, zorder=5, alpha=0.9)
    ax.annotate(name, (fps, mAP), textcoords='offset points',
                xytext=(12, 4), fontsize=11, fontweight='bold', color=color)

ax.set_xlabel('Inference Speed (FPS)', fontsize=12)
ax.set_ylabel('mAP@50', fontsize=12)
ax.set_title('Speed vs Accuracy Trade-off', fontsize=14, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
ax.set_xlim(left=0)
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/speed_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Training loss curves
Run this cell after training completes and `history.json` is saved.

In [ ]:
history_path = '../models/weights/fashionnet/history.json'
with open(history_path) as f:
    history = json.load(f)

epochs    = [h['epoch']     for h in history]
train_loss= [h['loss']      for h in history]
val_loss  = [h['val_loss']  for h in history]
box_loss  = [h['box']       for h in history]
obj_loss  = [h['obj']       for h in history]
cls_loss  = [h['cls']       for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('FashionNet Training History', fontsize=14, fontweight='bold')

axes[0].plot(epochs, train_loss, label='Train loss', color='#4C72B0')
axes[0].plot(epochs, val_loss,   label='Val loss',   color='#DD8452', linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss'); axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

axes[1].plot(epochs, box_loss, label='Box')
axes[1].plot(epochs, obj_loss, label='Objectness')
axes[1].plot(epochs, cls_loss, label='Class')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Components'); axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../docs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()